[![Mở trong Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/trungnguyenvn/mlsysbook-vi-notebooks/blob/main/vol1-ch10-model-compression/02-chung-cat-va-hang-thap.ipynb)

# Chưng cất tri thức và phân rã hạng thấp: hai cách có được một mô hình dày nhỏ hơn

Cắt tỉa để lại một mô hình thưa, và chương 10 nhấn mạnh rằng số 0 chỉ có giá trị khi đường thực
thi bỏ qua được chúng. Sổ tay này đi theo hai lối khác, cả hai đều cho ra **phép toán dày nhỏ
hơn**, chạy được trên kernel thông thường:

* **chưng cất tri thức (knowledge distillation)** huấn luyện một mô hình trò nhỏ bắt chước phân
  phối đầu ra của một mô hình thầy lớn;
* **phân rã hạng thấp** thay một ma trận trọng số bằng tích của hai ma trận mảnh.

**Bạn sẽ làm:**
1. huấn luyện một mô hình thầy trên MNIST;
2. xem nhãn mềm đổi thế nào theo nhiệt độ, và đọc tri thức ẩn (dark knowledge) trong đó;
3. kiểm tra vì sao hàm mất mát chưng cất nhân với bình phương nhiệt độ;
4. so một mô hình trò tự học với cùng mô hình trò được chưng cất, rồi tìm trường hợp chưng cất không giúp;
5. phân rã tầng đầu của mô hình thầy bằng SVD, ở sáu hạng;
6. tính lại ví dụ ma trận 4096 nhân 4096 ở hạng 128 của chương, rồi đo nó trên CPU.

In [ ]:
#@title Chuẩn bị: thư viện, hạt giống và hai hàm đối chiếu với sách { display-mode: "form" }
# Chạy ô này trước. Nó không cài thêm gì: mọi thư viện dùng ở đây đều có sẵn trên Colab.
import math, os, random, re, sys, time

import matplotlib.pyplot as plt
import numpy as np
import torch

# Số luồng CPU. 0 để thư viện tự chọn (trên Colab là 2).
NB_THREADS = int(os.environ.get("NB_THREADS", "0"))
if NB_THREADS:
    torch.set_num_threads(NB_THREADS)
    try:
        torch.set_num_interop_threads(NB_THREADS)
    except RuntimeError:  # chỉ đặt được một lần mỗi phiên
        pass
    from threadpoolctl import threadpool_limits
    threadpool_limits(NB_THREADS)

SEED = 42


def dat_hat_giong(seed=SEED):
    """Cùng hạt giống thì cùng kết quả, mỗi lần chạy lại."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


dat_hat_giong()
DATA = os.environ.get("NB_DATA", "data")  # nơi tải dữ liệu về
NB_STRICT = os.environ.get("NB_STRICT") == "1"
plt.rcParams.update({"figure.figsize": (7, 4), "figure.dpi": 110,
                     "axes.grid": True, "grid.alpha": 0.3})

# Đọc một con số theo cách bảng số liệu của bài học viết nó (dấu chấm hay dấu phẩy).
import math
import re

_SUP = str.maketrans("⁰¹²³⁴⁵⁶⁷⁸⁹⁻⁺", "0123456789-+")
_MULT = {"tỷ": 1e9, "tỉ": 1e9, "triệu": 1e6, "nghìn": 1e3, "ngàn": 1e3,
         "K": 1e3, "M": 1e6, "B": 1e9}
_NUM = re.compile(
    r"(?<![\w.,])(-?)(\d+(?:[.,]\d+)*)"                      # sign, digits with separators
    r"(?:\s*[×x·]\s*10\s*(?:\^\{?\s*([-+]?\d+)\s*\}?|([⁻⁺]?[⁰¹²³⁴⁵⁶⁷⁸⁹]+))|[eE]([-+]?\d+)"
    r"|\^\{?\s*([-+]?\d+)\s*\}?|([⁻⁺]?[⁰¹²³⁴⁵⁶⁷⁸⁹]+))?"                  # or a bare power, 10^{-5}
    r"(\s*%|\s*(?:tỷ|tỉ|triệu|nghìn|ngàn)(?!\w)|\s?[KMB](?![A-Za-z]))?")


def norm(text):
    """The outline's markup and typography reduced to plain text, as both sides compare it."""
    text = text.replace("{,}", ",").replace("**", "").replace("`", "")
    text = re.sub(r"[   ]", " ", text)
    text = re.sub(r"−(?=\d)", "-", text)
    text = re.sub(r"[~≈]", "", text)
    return re.sub(r"\s+", " ", text).strip()


def _readings(digits):
    """(value, decimals) for each way the separators in `digits` can be read."""
    out = []
    groups_ok = lambda parts: all(len(p) == 3 for p in parts[1:])
    for thou, dec in ((",", "."), (".", ",")):
        if thou in digits and dec in digits:
            head, _, tail = digits.rpartition(dec)
            if dec in head or not tail.isdigit() or not groups_ok(head.split(thou)):
                continue
            out.append((float(head.replace(thou, "") + "." + tail), len(tail)))
    for sep in (",", "."):
        other = "." if sep == "," else ","
        if sep in digits and other not in digits:
            parts = digits.split(sep)
            if groups_ok(parts):
                out.append((float("".join(parts)), 0))                    # thousands
            if len(parts) == 2:
                out.append((float(parts[0] + "." + parts[1]), len(parts[1])))  # decimal
    if "." not in digits and "," not in digits:
        out.append((float(digits), 0))
    return out


def candidates(text):
    """Every (value, half_unit) the numbers in `text` can mean.

    half_unit is half of the last printed digit, scaled like the value, so `437,5 KB`
    accepts 437.544. Each number is offered as written and, when a multiplier or `%`
    follows it, scaled by that too, so `8,5 tỉ` cites 8.5 or 8.5e9 and `20 %` cites 20 or 0.2.
    """
    out = []
    for m in _NUM.finditer(norm(text)):
        sign, digits, exp_caret, exp_sup, exp_e, pow_caret, pow_sup, suffix = m.groups()
        exp = exp_caret or (exp_sup.translate(_SUP) if exp_sup else None) or exp_e
        power = pow_caret or (pow_sup.translate(_SUP) if pow_sup else None)
        scale10 = 10.0 ** int(exp) if exp else 1.0
        suffix = (suffix or "").strip()
        extra = 0.01 if suffix == "%" else _MULT.get(suffix)
        for value, decimals in _readings(digits):
            if power:
                # 10^{-5}, 2³²: an exact value, so it is matched to 1 part in 10⁹; a looser
                # comparison needs an explicit tol.
                v = value ** int(power)
                half = abs(v) * 1e-9
            else:
                v = value * scale10
                half = 0.5 * 10.0 ** -decimals * scale10
            v = -v if sign else v
            out.append((v, half))
            if extra:
                out.append((v * extra, half * extra))
    return out


def match(value, text):
    """The (value, half_unit) reading of `text` that `value` equals, or None."""
    for v, half in candidates(text):
        if math.isclose(value, v, rel_tol=1e-9, abs_tol=1e-300):
            return v, half
    return None


def well_anchored(text):
    """False for a citation too bare to identify one cell, like "1" or "2"."""
    t = norm(text)
    rest = re.sub(r"[\d.,\s\-+]", "", t)
    if rest:
        return True
    sig = re.sub(r"[^\d]", "", t).lstrip("0")
    return len(sig) >= 2


def _so(x):
    """Một con số để in: số nguyên có dấu phẩy ngăn nghìn, số thực giữ 6 chữ số có nghĩa."""
    if isinstance(x, (int, np.integer)) or (isinstance(x, float) and x.is_integer() and abs(x) < 1e15):
        return f"{int(x):,}"
    return f"{x:,.6g}"


def sach(gia_tri, nguon=None, trich=None, tol=None):
    """Một con số của sách, kèm đúng chữ của nó: trong bảng số liệu của bài học (nguon)
    hoặc nguyên văn trong chương (trich)."""
    chu = nguon if nguon is not None else trich
    if match(gia_tri, chu) is None:
        raise ValueError(f"{gia_tri!r} không phải con số ghi trong {chu!r}")
    return gia_tri


def theo_sach(ten, gia_tri, sach, nguon=None, trich=None, tol=None):
    """So một con số tính được với con số của sách. In ✓ khi khớp, ✗ kèm độ lệch khi không."""
    chu = nguon if nguon is not None else trich
    khop = match(sach, chu)
    if khop is None:
        raise ValueError(f"{sach!r} không phải con số ghi trong {chu!r}")
    sai_so = tol if tol is not None else khop[1]
    dung = abs(gia_tri - sach) <= sai_so
    print(f"{'✓' if dung else '✗'} {ten}: tính được {_so(gia_tri)} · sách: {chu}")
    if not dung:
        loi = f"lệch {_so(gia_tri - sach)}, quá sai số cho phép {_so(sai_so)}"
        if NB_STRICT:
            raise AssertionError(f"{ten}: {loi}")
        print(f"   {loi}. Nếu bạn vừa đổi tham số ở trên thì đây là điều được chờ đợi.")
    # None, not the bool: a cell ending in theo_sach(...) would otherwise echo True under the ✓.


def do_tren_may(ten, gia_tri, don_vi=""):
    """Một con số ĐO trên máy đang chạy: mỗi máy một khác, không phải con số của sách."""
    print(f"⏱ {ten}: {_so(gia_tri)} {don_vi} (đo trên máy này)".replace("  ", " "))


print(f"Python {sys.version.split()[0]} · PyTorch {torch.__version__} · NumPy {np.__version__}"
      f" · {torch.get_num_threads()} luồng CPU")

## 1. Mô hình thầy

Mô hình thầy là một perceptron nhiều tầng (multilayer perceptron) 784-512-256-10, học 8 epoch
trên 10,000 ảnh huấn luyện. Độ chính xác đo trên cả 10,000 ảnh kiểm tra.

In [ ]:
import copy

import torch.nn.functional as F
from torchvision import datasets


def nap_mnist(train, n):
    ds = datasets.MNIST(DATA, train=train, download=True)
    x = ds.data[:n].float().div(255).view(-1, 784)
    return (x - 0.1307) / 0.3081, ds.targets[:n]   # chuẩn hoá theo trung bình và độ lệch của MNIST


X, y = nap_mnist(True, 10_000)
X_kt, y_kt = nap_mnist(False, 10_000)
KIEN_TRUC_THAY = (784, 512, 256, 10)
KIEN_TRUC_TRO = (784, 64, 10)


def mlp(*kich_thuoc):
    tang = []
    for a, b in zip(kich_thuoc[:-1], kich_thuoc[1:]):
        tang += [torch.nn.Linear(a, b), torch.nn.ReLU()]
    return torch.nn.Sequential(*tang[:-1])


def so_tham_so(m):
    return sum(p.numel() for p in m.parameters())


def do_chinh_xac(m):
    with torch.no_grad():
        return (m(X_kt).argmax(1) == y_kt).float().mean().item()


def huan_luyen(m, Xh, yh, so_epoch, thay=None, T=4.0, gamma=0.9, lr=1e-3):
    """Adam, batch 64. Có `thay` thì mất mát là mất mát chưng cất ở phần 3."""
    opt = torch.optim.Adam(m.parameters(), lr)
    if thay is not None:
        with torch.no_grad():
            logit_thay = thay(Xh)
    for _ in range(so_epoch):
        thu_tu = torch.randperm(len(Xh))
        for i in range(0, len(Xh), 64):
            chon = thu_tu[i:i + 64]
            out = m(Xh[chon])
            if thay is None:
                loss = F.cross_entropy(out, yh[chon])
            else:
                loss = mat_mat_chung_cat(out, logit_thay[chon], yh[chon], T, gamma)
            opt.zero_grad()
            loss.backward()
            opt.step()
    return m


def mat_mat_chung_cat(logit_tro, logit_thay, nhan, T, gamma):
    """(1 − γ)·CE(trò, nhãn cứng) + γ·T²·KL(thầy_T ‖ trò_T), cả hai phân phối cùng nhiệt độ T."""
    ce = F.cross_entropy(logit_tro, nhan)
    kl = F.kl_div(F.log_softmax(logit_tro / T, dim=1), F.softmax(logit_thay / T, dim=1),
                  reduction="batchmean")
    return (1 - gamma) * ce + gamma * T * T * kl


dat_hat_giong()
t0 = time.perf_counter()
thay = huan_luyen(mlp(*KIEN_TRUC_THAY), X, y, 8)
do_tren_may("thời gian huấn luyện mô hình thầy", time.perf_counter() - t0, "s")
acc_thay = do_chinh_xac(thay)
do_tren_may("độ chính xác của mô hình thầy", 100 * acc_thay, "%")
print(f"mô hình thầy có {so_tham_so(thay):,} tham số")
assert so_tham_so(thay) == 784 * 512 + 512 + 512 * 256 + 256 + 256 * 10 + 10

## 2. Nhãn mềm và nhiệt độ

Với một ảnh chữ số, nhãn cứng one-hot chỉ nói một điều: lớp đúng là lớp nào. Mô hình thầy thì
cho một phân phối trên cả mười lớp. Chương làm mềm phân phối ấy bằng một nhiệt độ $T$ trong
softmax:

$$p_i(T) = \frac{\exp(z_i / T)}{\sum_j \exp(z_j / T)}$$

Trước hết, viết công thức bằng tay và so với `torch.softmax`.

In [ ]:
def softmax_T(z, T):
    e = torch.exp((z - z.max()) / T)      # trừ max để không tràn số; kết quả không đổi
    return e / e.sum()


with torch.no_grad():
    logit_kt = thay(X_kt)
for T in (1.0, 4.0, 10.0):
    assert torch.allclose(softmax_T(logit_kt[0], T), torch.softmax(logit_kt[0] / T, dim=0), atol=1e-6)
print("công thức viết tay khớp torch.softmax ở T = 1, 4, 10")

### Dự đoán

Ta chọn ảnh kiểm tra mà mô hình thầy đoán đúng nhưng phân vân nhất: xác suất của lớp đứng thứ hai
cao nhất. Khi tăng $T$ từ 1 lên 10, xác suất của lớp đúng sẽ thế nào, và thứ tự của các lớp sai
có đổi không?

In [ ]:
xs = torch.softmax(logit_kt, dim=1)
dung = xs.argmax(1) == y_kt
hang_hai = xs.topk(2, dim=1).values[:, 1]
i_anh = int(torch.where(dung, hang_hai, torch.zeros_like(hang_hai)).argmax())
z = logit_kt[i_anh]

fig, (ax0, ax1) = plt.subplots(1, 2, figsize=(8, 3.2), gridspec_kw={"width_ratios": [1, 3]})
ax0.imshow((X_kt[i_anh] * 0.3081 + 0.1307).view(28, 28), cmap="gray_r")
ax0.set_title(f"ảnh kiểm tra số {i_anh}\nnhãn đúng: {int(y_kt[i_anh])}", fontsize=9)
ax0.axis("off")
rong = 0.27
for j, (T, mau) in enumerate([(1, "C0"), (4, "C1"), (10, "C2")]):
    ax1.bar(np.arange(10) + (j - 1) * rong, softmax_T(z, T).numpy(), rong, color=mau, label=f"T = {T}")
ax1.set_xticks(range(10))
ax1.set_xlabel("lớp")
ax1.set_ylabel("xác suất")
ax1.set_title("Nhãn mềm của mô hình thầy ở ba nhiệt độ", fontsize=10)
ax1.legend(loc="upper left", bbox_to_anchor=(1.01, 1), frameon=False)
plt.tight_layout()
plt.show()

thu_tu = [z.argsort(descending=True).tolist()]
print("thứ tự các lớp, từ xác suất cao xuống thấp:", thu_tu[0])
for T in (4.0, 10.0):
    assert softmax_T(z, T).argsort(descending=True).tolist() == thu_tu[0]  # T không đổi thứ tự


def entropy(p):
    return -(p * torch.log(p.clamp_min(1e-12))).sum(dim=1)


cac_T = [1, 2, 4, 8, 16]
H = [entropy(torch.softmax(logit_kt / T, dim=1)).mean().item() for T in cac_T]
for T, h in zip(cac_T, H):
    print(f"T = {T:>2}: entropy trung bình {h:.3f} nat (tối đa ln 10 = {math.log(10):.3f})")
assert all(a < b for a, b in zip(H, H[1:]))

Nhiệt độ không đổi thứ tự các lớp, vì chia mọi logit cho cùng một số dương giữ nguyên thứ tự của
chúng. Nó chỉ đổi **khoảng cách**: ở $T = 1$ hầu hết các lớp sai gần như bằng 0, ngoài lớp đứng
thứ hai, còn ở $T = 10$ chúng hiện ra đủ lớn để mô hình trò học được. Entropy trung bình tăng đều
theo $T$.

### Tri thức ẩn: các lớp sai không sai như nhau

Lấy trung bình nhãn mềm ở $T = 4$ trên mọi ảnh kiểm tra của cùng một chữ số, rồi bỏ đường chéo.
Mỗi ô cho biết mô hình thầy, khi nhìn chữ số ở hàng, dành bao nhiêu xác suất cho chữ số ở cột.

In [ ]:
p4 = torch.softmax(logit_kt / 4, dim=1)
ma_tran = torch.stack([p4[y_kt == c].mean(0) for c in range(10)])
ma_tran.fill_diagonal_(float("nan"))

fig, ax = plt.subplots(figsize=(5.2, 4.4))
anh = ax.imshow(ma_tran.numpy(), cmap="Blues")
ax.set_xticks(range(10))
ax.set_yticks(range(10))
ax.set_xlabel("lớp sai được gán xác suất")
ax.set_ylabel("chữ số thật")
ax.set_title("Tri thức ẩn: xác suất trung bình ở T = 4\n(bỏ đường chéo)", fontsize=10)
fig.colorbar(anh, ax=ax, fraction=0.046)
plt.show()

phang = torch.nan_to_num(ma_tran, nan=-1.0).flatten()
for k in phang.argsort(descending=True)[:4].tolist():
    print(f"chữ số {k // 10} được thầy xem gần với {k % 10} nhất: {phang[k]:.3f}")

Những cặp sáng nhất là những cặp mà người viết tay cũng hay làm lẫn. Nhãn cứng không mang thông
tin ấy, còn nhãn mềm thì mang. Đó là tín hiệu mà chương gọi là tri thức ẩn, phong phú hơn nhiều
so với một nhãn chỉ nói đúng hay sai.

## 3. Vì sao nhân với $T^2$

Hàm mất mát chưng cất của chương là

$$\mathcal L = (1-\gamma)\,\mathrm{CE}(\text{trò}, y) + \gamma\,T^2\,\mathrm{KL}\big(p^{\text{thầy}}(T)\,\|\,p^{\text{trò}}(T)\big)$$

Chương nói thừa số $T^2$ bù cho việc nhiệt độ làm giảm độ lớn của gradient. Ta đo điều đó: lấy
gradient của số hạng KL theo logit của một mô hình trò chưa học, ở nhiều nhiệt độ, có và không có
thừa số $T^2$.

### Dự đoán

Không có $T^2$, gradient nhỏ đi bao nhiêu lần khi $T$ tăng từ 4 lên 16?

In [ ]:
dat_hat_giong()
tro_chua_hoc = mlp(*KIEN_TRUC_TRO)
xb = X[:256]
with torch.no_grad():
    lt = thay(xb)
cac_T = [1, 2, 4, 8, 16]
khong_T2, co_T2 = [], []
for T in cac_T:
    zs = tro_chua_hoc(xb).detach().requires_grad_(True)
    kl = F.kl_div(F.log_softmax(zs / T, 1), F.softmax(lt / T, 1), reduction="batchmean")
    (g,) = torch.autograd.grad(kl, zs)
    khong_T2.append(g.norm().item())
    co_T2.append(T * T * g.norm().item())
for T, a, b in zip(cac_T, khong_T2, co_T2):
    print(f"T = {T:>2}: chuẩn gradient {a:.2e} không có T², {b:.2e} có T²")
print(f"từ T = 4 lên 16, không có T² gradient nhỏ đi {khong_T2[2] / khong_T2[4]:.1f} lần; (16/4)² = 16")
print(f"từ T = 1 lên 2, chỉ nhỏ đi {khong_T2[0] / khong_T2[1]:.1f} lần: quy luật 1/T² chỉ đúng khi T đủ lớn")

fig, ax = plt.subplots()
ax.loglog(cac_T, khong_T2, "o-", base=2, label="KL, không có T²")
ax.loglog(cac_T, co_T2, "s-", base=2, label="T² · KL")
ax.set_xticks(cac_T, [str(T) for T in cac_T])
ax.minorticks_off()
ax.set_xlabel("nhiệt độ T")
ax.set_ylabel("chuẩn gradient theo logit của trò")
ax.set_title("Khi T đủ lớn, thừa số T² giữ độ lớn gradient gần như không đổi", fontsize=10)
ax.legend(loc="lower left")
plt.show()

Khi $T$ lớn so với khoảng cách giữa các logit, gradient của KL theo logit của trò xấp xỉ
$(z^{\text{trò}} - z^{\text{thầy}})/(N T^2)$, với $N$ là số lớp, nên nó nhỏ đi theo $T^2$. Nhân với
$T^2$ đưa nó về một độ lớn gần như không đổi, và nhờ vậy $\gamma$ giữ nguyên ý nghĩa cân bằng giữa
hai số hạng khi bạn đổi nhiệt độ. Ở $T$ nhỏ xấp xỉ ấy chưa đúng, nên quy luật $1/T^2$ chỉ hiện rõ
từ khoảng $T = 4$ trở đi.

## 4. Mô hình trò: tự học, hay học từ thầy

Mô hình trò là 784-64-10, nhỏ hơn mô hình thầy khoảng mười lần. **Giả định của sổ tay:** mô hình
thầy đã học từ cả 10,000 ảnh, còn mô hình trò chỉ được học trên 2,000 ảnh trong số đó, 20 epoch.
Cùng ảnh, cùng số bước, cùng khởi tạo, chỉ khác hàm mất mát: nhãn cứng một mình, hay mất mát chưng
cất với $\gamma = 0.9$ ở bốn nhiệt độ. Mỗi cấu hình chạy với hai hạt giống.

### Dự đoán

Mô hình trò được chưng cất có hơn mô hình trò tự học không? Nếu có, ở nhiệt độ nào?

In [ ]:
X_tro, y_tro = X[:2000], y[:2000]
SO_EPOCH_TRO = 20
HAT_GIONG = [0, 1]
print(f"mô hình trò có {so_tham_so(mlp(*KIEN_TRUC_TRO)):,} tham số,"
      f" nhỏ hơn mô hình thầy {so_tham_so(thay) / so_tham_so(mlp(*KIEN_TRUC_TRO)):.1f} lần")


def chay_tro(thay_, T=None, Xh=X_tro, yh=y_tro, so_epoch=SO_EPOCH_TRO, ca_hai=False):
    """Độ chính xác trung bình qua các hạt giống (và độ lệch giữa chúng nếu ca_hai)."""
    kq = []
    for hg in HAT_GIONG:
        dat_hat_giong(100 + hg)
        tro = mlp(*KIEN_TRUC_TRO)
        if T is None:
            huan_luyen(tro, Xh, yh, so_epoch)
        else:
            huan_luyen(tro, Xh, yh, so_epoch, thay=thay_, T=T)
        kq.append(do_chinh_xac(tro))
    return (float(np.mean(kq)), max(kq) - min(kq)) if ca_hai else float(np.mean(kq))


t0 = time.perf_counter()
acc_tu_hoc, lech_hat_giong = chay_tro(None, ca_hai=True)
nhiet_do = [1, 2, 4, 8]
acc_chung_cat = [chay_tro(thay, T) for T in nhiet_do]
do_tren_may("thời gian cho 10 lần huấn luyện mô hình trò", time.perf_counter() - t0, "s")
do_tren_may("trò tự học, trung bình hai hạt giống", 100 * acc_tu_hoc, "%")
do_tren_may("trò tự học, chênh lệch giữa hai hạt giống", 100 * lech_hat_giong, "điểm phần trăm")
for T, a in zip(nhiet_do, acc_chung_cat):
    do_tren_may(f"trò được chưng cất, T = {T}", 100 * a, "%")

fig, ax = plt.subplots()
ax.plot(nhiet_do, [100 * a for a in acc_chung_cat], "o-", color="C1", label="trò được chưng cất")
ax.axhline(100 * acc_tu_hoc, color="C0", label="trò tự học (nhãn cứng)")
ax.axhline(100 * acc_thay, color="0.5", ls="--", label="mô hình thầy")
ax.set_xscale("log", base=2)
ax.set_xticks(nhiet_do, [str(T) for T in nhiet_do])
ax.minorticks_off()
ax.set_xlabel("nhiệt độ T")
ax.set_ylabel("độ chính xác trên tập kiểm tra (%)")
ax.set_title(f"Mô hình trò {'-'.join(map(str, KIEN_TRUC_TRO))}, học trên 2,000 ảnh")
ax.legend(loc="center left")
plt.show()

Nếu đường cam nằm trên đường xanh, phần chênh ấy là thứ mô hình trò nhận được từ thầy mà nhãn
cứng của 2,000 ảnh không có. Hai hạt giống chỉ đủ để thấy chiều của hiệu ứng: hãy so phần chênh
với độ lệch giữa hai hạt giống in ở trên. Một khác biệt cỡ độ lệch ấy chưa nói lên điều gì.

## 5. Khi chưng cất không giúp

Chương nói thẳng rằng chưng cất không phải nén không mất mát, và hiệu quả tuỳ chất lượng mô hình
thầy cùng dung lượng mô hình trò. Hai thí nghiệm kiểm tra điều đó:

1. **mô hình thầy chỉ biết bằng mô hình trò**: một mô hình thầy cùng kiến trúc, nhưng chỉ học
   trên đúng 2,000 ảnh của mô hình trò;
2. **mô hình trò đã có đủ nhãn**: mô hình trò được học trên cả 10,000 ảnh, 8 epoch.

### Dự đoán

Trong trường hợp nào phần lợi của chưng cất còn lại?

In [ ]:
dat_hat_giong(7)
thay_yeu = huan_luyen(mlp(*KIEN_TRUC_THAY), X_tro, y_tro, SO_EPOCH_TRO)
do_tren_may("mô hình thầy chỉ học 2,000 ảnh", 100 * do_chinh_xac(thay_yeu), "%")
for T in (4, 8):
    do_tren_may(f"trò chưng cất từ thầy ấy, T = {T}", 100 * chay_tro(thay_yeu, T), "%")
print()
do_tren_may("trò tự học trên 10,000 ảnh", 100 * chay_tro(None, Xh=X, yh=y, so_epoch=8), "%")
do_tren_may("trò chưng cất trên 10,000 ảnh, T = 8", 100 * chay_tro(thay, 8, Xh=X, yh=y, so_epoch=8), "%")
print()
with torch.no_grad():
    for ten_thay, m_thay in [("thầy học 10,000 ảnh", thay), ("thầy chỉ học 2,000 ảnh", thay_yeu)]:
        h4 = entropy(torch.softmax(m_thay(X_tro) / 4, dim=1)).mean().item()
        do_tren_may(f"entropy nhãn mềm ở T = 4 trên ảnh của trò, {ten_thay}", h4, "nat")

So hai dòng chưng cất từ mô hình thầy yếu với trò tự học ở phần 4, rồi nhìn hai dòng entropy cuối.
Nếu hai giá trị entropy gần nhau, nhãn mềm của hai mô hình thầy mềm như nhau trên ảnh của trò, và
điều khác nhau không phải độ mềm mà là **nội dung**: mô hình thầy yếu không biết gì ngoài chính
2,000 ảnh ấy, nên xác suất nó dành cho các lớp sai không dạy được điều mà nhãn cứng của chúng chưa
nói.

Ở hai dòng giữa, mô hình trò đã có đủ nhãn. Khi đó thứ giới hạn nó là dung lượng của chính nó, và
nhãn mềm không thêm được dung lượng, có khi còn làm nó kém đi. Chưng cất dời chi phí sang một pha
huấn luyện thêm, và pha ấy chỉ đáng khi mô hình thầy thực sự biết nhiều hơn điều mô hình trò tự học
được.

## 6. Phân rã hạng thấp một tầng đã huấn luyện

Phân rã hạng thấp xấp xỉ một ma trận $\mathbf A$ cỡ $m \times n$ bằng $\mathbf U \mathbf V^T$, với
$\mathbf U$ cỡ $m \times k$ và $\mathbf V$ cỡ $n \times k$. Cách tính thường dùng là SVD, giữ $k$
giá trị suy biến lớn nhất. Lưu trữ giảm từ $mn$ xuống $k(m+n)$.

Ta phân rã tầng đầu của mô hình thầy, ma trận 512 nhân 784. Chương nói xấp xỉ có chính xác hay
không tuỳ phổ giá trị suy biến, nên ta so phổ ấy với phổ của một ma trận ngẫu nhiên cùng cỡ.

### Dự đoán

Giữ hạng 32 (chưa tới một phần mười số tham số của tầng), độ chính xác của mô hình thầy còn bao
nhiêu?

In [ ]:
W1 = thay[0].weight.detach()              # 512 x 784
m_, n_ = W1.shape
U, S, Vh = torch.linalg.svd(W1, full_matrices=False)
dat_hat_giong()
R = torch.randn(m_, n_) * W1.std()
S_ngau_nhien = torch.linalg.svdvals(R)

fig, ax = plt.subplots()
ax.semilogy(np.arange(1, len(S) + 1), (S / S[0]).numpy(), label="tầng đầu của mô hình thầy")
ax.semilogy(np.arange(1, len(S) + 1), (S_ngau_nhien / S_ngau_nhien[0]).numpy(), label="ma trận ngẫu nhiên cùng cỡ")
ax.set_xlabel("thứ tự giá trị suy biến")
ax.set_ylabel("giá trị suy biến / giá trị lớn nhất (thang log)")
ax.set_title("Phổ giá trị suy biến của ma trận 512 nhân 784")
ax.legend(loc="lower left")
plt.show()

cac_hang = [8, 16, 32, 64, 128, 256]
sai_so, acc_hang, ti_le_ts = [], [], []
for k in cac_hang:
    Uk, Vk = U[:, :k] * S[:k], Vh[:k].T          # A ≈ Uk · Vkᵀ
    sai_so.append(((W1 - Uk @ Vk.T).norm() / W1.norm()).item())
    # Định lý Eckart–Young: sai số của SVD cắt ngắn đúng bằng phần năng lượng bị bỏ
    assert abs(sai_so[-1] - (S[k:].square().sum() / S.square().sum()).sqrt().item()) < 1e-4
    # Thay tầng đầu bằng hai tầng tuyến tính: 784 → k (không độ lệch), k → 512 (giữ độ lệch cũ)
    ha = torch.nn.Linear(n_, k, bias=False)
    hb = torch.nn.Linear(k, m_)
    with torch.no_grad():
        ha.weight.copy_(Vk.T)
        hb.weight.copy_(Uk)
        hb.bias.copy_(thay[0].bias)
    moi = torch.nn.Sequential(ha, hb, *copy.deepcopy(list(thay)[1:]))
    assert ha.weight.numel() + hb.weight.numel() == k * (m_ + n_)
    ti_le_ts.append(k * (m_ + n_) / (m_ * n_))
    acc_hang.append(do_chinh_xac(moi))

for k, f, e, a in zip(cac_hang, ti_le_ts, sai_so, acc_hang):
    print(f"hạng {k:>3}: {f:6.1%} tham số của tầng · sai số tương đối {e:.3f} · độ chính xác {a:.2%}")

fig, ax = plt.subplots()
ax.plot([100 * f for f in ti_le_ts], [100 * a for a in acc_hang], "o-", label="tầng đầu phân rã")
ax.axhline(100 * acc_thay, color="0.5", ls="--", label="mô hình thầy, tầng dày")
for k, f, a in zip(cac_hang, ti_le_ts, acc_hang):
    ax.annotate(f"k = {k}", (100 * f, 100 * a), xytext=(7, -4) if k == cac_hang[0] else (4, -13),
                textcoords="offset points", fontsize=8)
ax.set_xscale("log")
ax.set_xlim(2, 140)
ax.set_xticks([100 * f for f in ti_le_ts], [f"{100 * f:.0f}" for f in ti_le_ts])
ax.minorticks_off()
ax.set_xlabel("tham số của tầng đầu, so với ma trận dày (%, thang log)")
ax.set_ylabel("độ chính xác trên tập kiểm tra (%)")
ax.set_title("Phân rã hạng thấp tầng 512 nhân 784, không tinh chỉnh")
ax.legend(loc="lower right")
plt.show()

Sai số tương đối của ma trận còn khá lớn ở hạng thấp, mà độ chính xác đã gần như đủ. Hai đại lượng
ấy không giống nhau: phần bị bỏ là những hướng mà dữ liệu hầu như không đi qua. Ma trận ngẫu nhiên
thì không có hướng nào như thế, phổ của nó phẳng, nên hạng thấp cắt mất phần lớn của nó. Phân rã
chỉ đáng khi trọng số thực sự dư thừa về mặt toán học, và điều đó phải đo trên ma trận thật.

## 7. Ma trận 4096 nhân 4096 ở hạng 128

Góc nhìn hệ thống 1.2 của chương: lưu một ma trận 4096 nhân 4096 ở FP32 cần 67.1 MB, còn hai
thừa số hạng 128 chỉ cần 4.2 MB. Sách tính megabyte theo $10^6$ byte, và gọi đây là ví dụ minh hoạ.

In [ ]:
m = n = sach(4096, nguon="4096 nhân 4096")
k = sach(128, nguon="hạng 128")
BYTE_FP32 = 4

mb_day = m * n * BYTE_FP32 / 1e6
mb_thua_so = (m * k + k * n) * BYTE_FP32 / 1e6
theo_sach("ma trận dày FP32 (MB)", mb_day, sach=67.1, nguon="67.1 MB")
theo_sach("hai thừa số hạng 128 (MB)", mb_thua_so, sach=4.2, nguon="67.1 MB · 4.2 MB")
theo_sach("số lần giảm giá trị lưu trữ", (m * n) / (m * k + k * n), sach=16, nguon="4.2 MB · 16×")
theo_sach("tỉ lệ tham số còn lại (%)", 100 * (m * k + k * n) / (m * n), sach=6.25, nguon="6.25 phần trăm")

mac_day = m * n
mac_thua_so = k * (m + n)
mac_tao_lai = m * k * n
print(f"MAC một phép nhân với vector: dày {mac_day:,} · hai thừa số {mac_thua_so:,}"
      f" ({mac_day / mac_thua_so:.0f} lần ít hơn)")
print(f"tính tường minh U·Vᵀ trước: {mac_tao_lai:,} MAC, gấp {mac_tao_lai / mac_day:.0f} lần chính phép nhân dày")

Vì $m = n$, số phép nhân-cộng giảm đúng bằng số giá trị lưu trữ, 16 lần. Còn nếu dựng lại
$\mathbf U\mathbf V^T$ trước khi nhân, ta trả $mkn$ phép tính, gấp 128 lần chính phép nhân dày, và
mất hết mục đích.

### Dự đoán

Với một vector đầu vào (batch 1), phép nhân qua hai thừa số nhanh hơn phép nhân dày bao nhiêu lần
trên CPU này: đúng 16, ít hơn, hay nhiều hơn?

In [ ]:
def do_thoi_gian(f, lap=30):
    for _ in range(3):                       # khởi động, không tính
        f()
    ts = []
    for _ in range(lap):
        t0 = time.perf_counter()
        f()
        ts.append(time.perf_counter() - t0)
    return float(np.median(ts))


dat_hat_giong()
A = torch.randn(m, n)
Uf, Vf = torch.randn(m, k), torch.randn(k, n)       # chỉ cần hình dạng, không cần giá trị
v = torch.randn(n)

t_day = do_thoi_gian(lambda: torch.mv(A, v))
t_thua_so = do_thoi_gian(lambda: torch.mv(Uf, torch.mv(Vf, v)))
t_tao_lai = do_thoi_gian(lambda: torch.mv(Uf @ Vf, v), lap=3)
do_tren_may("nhân dày 4096 nhân 4096 với vector", 1e3 * t_day, "ms")
do_tren_may("qua hai thừa số hạng 128", 1e3 * t_thua_so, "ms")
do_tren_may("dựng lại U·Vᵀ rồi mới nhân", 1e3 * t_tao_lai, "ms")
do_tren_may("tỉ số dày / hai thừa số", t_day / t_thua_so, "lần")

fig, ax = plt.subplots(figsize=(7, 3))
ten = ["dựng lại U·Vᵀ rồi nhân", "nhân dày", "hai thừa số hạng 128"]
gia_tri = [1e3 * t_tao_lai, 1e3 * t_day, 1e3 * t_thua_so]
ax.barh(ten, gia_tri, color=["C3", "C0", "C1"])
ax.set_xscale("log")
for i, g in enumerate(gia_tri):
    ax.annotate(f"{g:.3g} ms", (g, i), xytext=(4, 0), textcoords="offset points", va="center", fontsize=9)
ax.set_xlim(right=max(gia_tri) * 8)
ax.set_xlabel("thời gian (ms, thang log)")
ax.set_title("Một phép nhân ma trận 4096 nhân 4096 với vector", fontsize=10)
plt.tight_layout()
plt.show()

Ở batch 1, phép nhân dày gần như chỉ là đọc 67.1 MB trọng số: nó nghẽn ở bộ nhớ (memory-bound),
và chương gọi đây là phiên bản cục bộ của tường bộ nhớ. Hai thừa số đọc ít hơn 16 lần, nhưng chúng
là hai phép toán, mỗi phép có phụ trội riêng, nên tỉ số đo được thường không đúng bằng 16. Đó là
câu hỏi triển khai của chương: mức giảm có đủ lớn để bù một phép toán thừa số thứ hai không.

## Thử thêm

1. Ở ô đầu tiên, đổi `KIEN_TRUC_TRO` thành `(784, 16, 10)` rồi chạy lại từ đầu. Mô hình trò càng
   nhỏ thì phần lợi của chưng cất lớn hơn hay nhỏ hơn?
2. Ở phần 4, đặt `gamma=0.5` trong `huan_luyen`. Cân lại giữa nhãn cứng và nhãn mềm đổi kết quả
   ra sao ở $T = 8$?
3. Ở phần 7, đổi `k` thành 512, rồi 2048. Từ hạng nào thì hai thừa số không còn rẻ hơn ma trận dày,
   cả về byte lẫn về thời gian? (Các dấu ✗ khi đó là điều được chờ đợi: sách tính với hạng 128.)

## Tóm lại

* Nhiệt độ không đổi thứ tự các lớp mà chỉ làm lộ khoảng cách giữa chúng, và tri thức ẩn là xác
  suất mô hình thầy dành cho những lớp sai gần nghĩa.
* Thừa số $T^2$ giữ gradient của số hạng KL ở cùng độ lớn khi đổi nhiệt độ.
* Chưng cất giúp khi mô hình thầy biết nhiều hơn điều mô hình trò tự học được, và gần như không giúp
  khi thầy chỉ thuộc đúng dữ liệu của trò, hay khi trò đã có đủ nhãn.
* Phân rã hạng thấp chỉ đáng khi phổ giá trị suy biến tắt nhanh. Ở hạng 128, ma trận 4096 nhân 4096
  lưu ít hơn 16 lần, nhưng thời gian thật còn tuỳ phụ trội của phép toán thứ hai.

---
### Nguồn và giấy phép

Sổ tay này đi kèm bài học [Nén mô hình — đổi phần dư thừa lấy ràng buộc triển khai](https://d3lu8vk4we0dc.cloudfront.net/lessons/vol1-ch10-model-compression/), dựng từ chương
[*Model Compression*](https://mlsysbook.ai/vol1/model_compression/model_compression.html) của sách.

Nội dung gốc thuộc Machine Learning Systems của Vijay Janapa Reddi và cộng sự, MIT Press. Bản quyền © 2024-2026 Harvard University. Giấy phép CC BY-NC-SA 4.0. Bản tiếng Việt này là tác phẩm phái sinh dùng cùng giấy phép: ghi công, chia sẻ tương tự, phi thương mại.